# StyleMatch · reusable author/source expansion

Adds 30–45 formally eligible author-language profiles without fine-tuning the deployed encoder. The notebook fetches curated original-language texts, discovers additional public-domain works through Gutendex/Project Gutenberg, includes the user-cleared Rowling GitHub text, rebuilds chunks and source-heldout splits, then builds a new versioned index. Existing production artifacts are not overwritten.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, shutil, subprocess, sys

drive.mount('/content/drive')
REPO = Path('/content/drive/MyDrive/style_matching')
assert (REPO / 'data/source_registry/expansion_authors_2026_07.csv').exists(), 'Pull the commit containing this notebook and candidate registry'
assert (REPO / 'scripts/evaluate_robust_rank_fusion.py').exists(), 'Update the repository before running this notebook'
os.chdir(REPO)

def run(cmd, check=True):
    print('>>>', ' '.join(map(str, cmd)), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if check and code:
        raise RuntimeError(f'command failed with exit {code}: {" ".join(map(str, cmd))}')
    return code

run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pandas', 'pyarrow', 'requests', 'beautifulsoup4', 'sentence-transformers==5.6.0', 'transformers==5.12.1', 'scikit-learn'])
run([sys.executable, '-c', "import importlib.metadata as m; assert m.version('sentence-transformers') == '5.6.0'; assert m.version('transformers') == '5.12.1'; import sentence_transformers.base; print('Model runtime:', m.version('sentence-transformers'), m.version('transformers'))"])
import pandas as pd

TARGET_FORMAL_ADDITIONS = 45
MIN_FORMAL_ADDITIONS = 30
RUN_RECALIBRATION = True
EXP = REPO / 'artifacts/source_expansion_v2'
EXP.mkdir(parents=True, exist_ok=True)

## 1 · Freeze the starting index and select curated missing profiles

The selection is computed from the actual index, not from assumptions about which authors are already online. Only distinct works count as independent sources.

In [ ]:
OLD_INDEX = REPO / 'artifacts/multilingual_style_index_challenger_v1'
NEW_INDEX = REPO / 'artifacts/multilingual_style_index_challenger_expanded_v2'
MODEL = REPO / 'artifacts/multilingual_author_style_v1'
assert (OLD_INDEX / 'profiles.parquet').exists()
assert (MODEL / 'training_config.json').exists()
before = pd.read_parquet(OLD_INDEX / 'profiles.parquet')
before_keys = set(zip(before['author_or_speaker'].astype(str), before['language'].astype(str)))

catalog = pd.read_csv(REPO / 'data/source_registry/multilingual_source_catalog.csv').fillna('')
catalog['profile_key'] = list(zip(catalog['name'].astype(str), catalog['original_language'].astype(str)))
counts = catalog.groupby('profile_key')['source_id'].nunique()
curated_keys = {key for key, count in counts.items() if count >= 3 and key not in before_keys}
curated = catalog[catalog['profile_key'].isin(curated_keys) & catalog['source_format'].ne('local_text')].drop(columns='profile_key')
curated = curated[curated.groupby(['name', 'original_language'])['source_id'].transform('nunique').ge(3)]

rowling = {
    'corpus': 'literary',
    'name': 'J. K. Rowling',
    'original_language': 'en',
    'year': '1997',
    'title': "Harry Potter and the Sorcerer's Stone",
    'source_id': 'github_rowling_sorcerers_stone',
    'source_url': 'https://raw.githubusercontent.com/amephraim/nlp/master/texts/J.%20K.%20Rowling%20-%20Harry%20Potter%201%20-%20Sorcerer%27s%20Stone.txt',
    'source_format': 'http_text',
    'independent_source_id': 'harry_potter_1_sorcerers_stone',
    'domain': 'literature',
    'register': 'literary_prose',
    'source_type': 'work',
    'delivered_language': 'en',
    'license_status': 'user_cleared_noncommercial_research',
    'display_allowed': 'false',
    'canonical_url': 'https://github.com/amephraim/nlp/blob/master/texts/J.%20K.%20Rowling%20-%20Harry%20Potter%201%20-%20Sorcerer%27s%20Stone.txt',
}
curated = pd.concat([curated, pd.DataFrame([rowling])], ignore_index=True).fillna('')
CURATED_CATALOG = EXP / 'curated_expansion_catalog.csv'
CURATED_MANIFEST = EXP / 'curated_expansion_manifest.csv'
curated.to_csv(CURATED_CATALOG, index=False)
print('curated profiles:', curated.groupby(['original_language', 'name']).ngroups)
display(curated.groupby(['original_language', 'name'])['source_id'].nunique().rename('sources').reset_index())

## 2 · Fetch curated texts, including Rowling

The Rowling source is marked user-cleared, noncommercial research and is excluded from passage display. It contributes one exploratory source unless two additional independent works are supplied later.

In [ ]:
fetch_code = run([sys.executable, 'scripts/fetch_multilingual_sources.py', '--catalog', str(CURATED_CATALOG), '--manifest', str(CURATED_MANIFEST), '--skip-existing'], check=False)
assert CURATED_MANIFEST.exists(), 'No curated source was downloaded'
run([sys.executable, 'scripts/import_source_manifest.py', str(CURATED_MANIFEST), '--append', '--skip-missing'])
if fetch_code:
    print('Some curated URLs failed; successful sources were retained and Gutendex discovery will fill the target.')

## 3 · Discover additional original-language works

The candidate pool contains 48 new French, German, Spanish, and Italian authors. Gutendex is used only as catalog metadata; downloaded text URLs and canonical records point to Project Gutenberg. Authors with fewer than three independently titled works are discarded.

In [ ]:
CANDIDATES = REPO / 'data/source_registry/expansion_authors_2026_07.csv'
for language in ('fr', 'de', 'es', 'it'):
    run([sys.executable, 'scripts/fetch_gutendex.py', '--corpus', 'literary', '--language', language, '--registry', str(CANDIDATES), '--batch', 'expansion_literary_non_en_2026_07', '--max-works', '3', '--min-works', '3'])

sources_path = REPO / 'data/literary/meta/sources.csv'
sources = pd.read_csv(sources_path).fillna('')
candidate_rows = pd.read_csv(CANDIDATES).fillna('')
candidate_names = set(candidate_rows['name'])
source_counts = sources[sources['author_or_speaker'].isin(candidate_names)].groupby(['author_or_speaker', 'language'])['independent_source_id'].nunique()
eligible = {(name, language) for (name, language), count in source_counts.items() if count >= 3}
print('Gutendex-eligible candidate profiles:', len(eligible))
display(source_counts.sort_values(ascending=False).rename('independent_sources').reset_index())

## 4 · Select 30–45 formal additions and update the registry

Selection is balanced round-robin across languages. Candidate source rows beyond the locked target are excluded from the rebuilt corpus.

In [ ]:
all_sources = []
for corpus in ('literary', 'rhetorical'):
    path = REPO / f'data/{corpus}/meta/sources.csv'
    if path.exists():
        part = pd.read_csv(path).fillna('')
        part['corpus'] = corpus
        all_sources.append(part)
source_frame = pd.concat(all_sources, ignore_index=True)
profile_source_counts = source_frame.groupby(['author_or_speaker', 'language'])['independent_source_id'].nunique()
curated_formal = {key for key, count in profile_source_counts.items() if count >= 3 and key not in before_keys and key not in eligible}
needed = max(0, TARGET_FORMAL_ADDITIONS - len(curated_formal))

queues = {language: sorted(name for name, lang in eligible if lang == language) for language in ('fr', 'de', 'es', 'it')}
selected = []
while len(selected) < needed and any(queues.values()):
    for language in ('fr', 'de', 'es', 'it'):
        if queues[language] and len(selected) < needed:
            selected.append((queues[language].pop(0), language))
selected_set = set(selected)
formal_additions = curated_formal | selected_set
assert MIN_FORMAL_ADDITIONS <= len(formal_additions) <= TARGET_FORMAL_ADDITIONS, f'Only {len(formal_additions)} formal additions; inspect the source table and add candidates before indexing'

keep_names = {name for name, _ in selected_set}
literary_sources = pd.read_csv(sources_path).fillna('')
drop_mask = literary_sources['author_or_speaker'].isin(candidate_names - keep_names)
literary_sources.loc[~drop_mask].to_csv(sources_path, index=False)

selected_registry = candidate_rows[candidate_rows.apply(lambda row: (row['name'], row['original_language']) in selected_set, axis=1)]
all_people_path = REPO / 'data/source_registry/all_people.csv'
all_people = pd.read_csv(all_people_path).fillna('')
all_people = pd.concat([all_people, selected_registry[all_people.columns]], ignore_index=True).drop_duplicates(['name', 'corpus', 'original_language'], keep='first')
all_people.to_csv(all_people_path, index=False)
literary_registry_path = REPO / 'data/source_registry/literary_authors.csv'
literary_registry = pd.read_csv(literary_registry_path).fillna('')
literary_registry = pd.concat([literary_registry, selected_registry[literary_registry.columns]], ignore_index=True).drop_duplicates(['name', 'corpus', 'original_language'], keep='first')
literary_registry.to_csv(literary_registry_path, index=False)
print('formal additions locked:', len(formal_additions))
display(pd.DataFrame(sorted(formal_additions), columns=['author', 'language']))

## 5 · Rebuild chunks and frozen splits

This changes corpus/index artifacts only. It does not run `finetune_multilingual_style.py`.

In [ ]:
run([sys.executable, 'scripts/audit_source_registry.py'])
CHUNKS = REPO / 'data/all/meta/all_sources_chunks.parquet'
COVERAGE = REPO / 'data/all/meta/all_sources_coverage.json'
HELDOUT = REPO / 'data/all/meta/all_source_heldout_splits.parquet'
HELDOUT_REPORT = REPO / 'data/all/meta/all_source_heldout_report.json'
run([sys.executable, 'scripts/build_chunk_parquet_from_sources.py', '--corpus', 'both', '--output', str(CHUNKS), '--coverage-output', str(COVERAGE), '--min-sources', '3', '--min-chunks', '30'])
run([sys.executable, 'scripts/make_source_heldout_splits.py', '--input', str(CHUNKS), '--output', str(HELDOUT), '--report', str(HELDOUT_REPORT)])
coverage = json.loads(COVERAGE.read_text())
display(pd.DataFrame(coverage['people']).query('source_heldout_ready == True').groupby('language').size().rename('formal_profiles'))

## 6 · Build a new index with the frozen challenger

Existing embedding caches are copied when available, so only unseen chunk IDs are encoded. The old production index remains intact.

In [ ]:
run([sys.executable, '-c', "import sentence_transformers.base; from sentence_transformers import SentenceTransformer; print('Sentence Transformers model modules: OK')"])
NEW_INDEX.mkdir(parents=True, exist_ok=True)
for filename in ('chunk_embeddings.npz', 'topic_chunk_embeddings.npz'):
    source = OLD_INDEX / filename
    target = NEW_INDEX / filename
    if source.exists() and not target.exists():
        shutil.copy2(source, target)
build = [sys.executable, 'scripts/multilingual_style_index.py', 'build', '--input', str(CHUNKS), '--out-dir', str(NEW_INDEX), '--model-name', str(MODEL), '--topic-model-name', 'intfloat/multilingual-e5-base', '--embedding-cache', str(NEW_INDEX / 'chunk_embeddings.npz'), '--topic-embedding-cache', str(NEW_INDEX / 'topic_chunk_embeddings.npz'), '--batch-size', '128', '--per-source-cap', '50', '--profile-cap', '600', '--profile-strategy', 'single_centroid', '--heldout-report', str(HELDOUT_REPORT), '--model-label', 'challenger_finetuned', '--artifact-version', 'challenger_v2_expanded', '--device', 'cuda']
comparison = REPO / 'artifacts/model_comparison_v1.json'
if comparison.exists():
    build.extend(['--model-comparison', str(comparison)])
run(build)

## 7 · Recalibrate and verify application readiness

In [ ]:
after = pd.read_parquet(NEW_INDEX / 'profiles.parquet')
after_keys = set(zip(after['author_or_speaker'].astype(str), after['language'].astype(str)))
added = after_keys - before_keys
formal_after = {(row['author_or_speaker'], row['language']) for row in coverage['people'] if row['source_heldout_ready']}
formal_added = formal_after - before_keys
assert MIN_FORMAL_ADDITIONS <= len(formal_added) <= TARGET_FORMAL_ADDITIONS, f'Unexpected formal profile addition count: {len(formal_added)}'
print('index profiles before/after/added:', len(before_keys), len(after_keys), len(added))

expanded_eval = EXP / 'expanded_source_heldout_eval'
run([sys.executable, 'scripts/style_embedding_recall.py', '--input', str(HELDOUT), '--out-dir', str(expanded_eval), '--model-name', str(MODEL), '--batch-size', '128', '--train-cap', '300', '--eval-splits', 'dev,test', '--device', 'cuda', '--skip-existing'])

if RUN_RECALIBRATION:
    calibration_dir = EXP / 'open_set_calibration'
    for language, count in after.groupby('language')['author_or_speaker'].nunique().items():
        if count >= 10:
            run([sys.executable, 'scripts/evaluate_open_set.py', '--input', str(HELDOUT), '--model-name', str(MODEL), '--out-dir', str(calibration_dir / language), '--language', language, '--device', 'cuda'])
    run([sys.executable, 'scripts/multilingual_style_index.py', 'calibrate', '--index-dir', str(NEW_INDEX), '--open-set-calibration-dir', str(calibration_dir)])

query = 'The institution changed slowly while ordinary people learned to live with its contradictions.'
run([sys.executable, 'scripts/multilingual_style_index.py', 'query', '--index-dir', str(NEW_INDEX), '--language', 'en', '--mode', 'cross', '--text', query, '--top-k', '3', '--device', 'cuda'])
summary = {
    'old_profiles': len(before_keys),
    'new_profiles': len(after_keys),
    'added_profiles': [{'name': name, 'language': language} for name, language in sorted(added)],
    'formal_added_profiles': [{'name': name, 'language': language} for name, language in sorted(formal_added)],
    'new_index': str(NEW_INDEX),
    'source_heldout_metrics': str(expanded_eval / 'style_embedding_metrics.json'),
    'model_retrained': False,
}
(EXP / 'expansion_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False))
print('RETURN THESE FILES:')
for path in [EXP / 'expansion_summary.json', COVERAGE, HELDOUT_REPORT, expanded_eval / 'style_embedding_metrics.json', NEW_INDEX / 'metadata.json']:
    print(path)
print('NEW INDEX:', NEW_INDEX)

## 8 · Upgrade the reranker on expanded independent-source evidence

This experiment does not alter the new index. It screens three deliberately distinct views: the deployed fine-tuned encoder, its pretrained counterpart, and classical style evidence. Scores are converted to within-language candidate percentiles, then combined with non-negative weights whose deployed-model weight cannot fall below 0.5. Selection uses dev sources only, a profile-cluster bootstrap lower bound, and language/corpus non-degradation constraints; test is opened once.

In [ ]:
RERANKER = EXP / 'robust_reranker_upgrade'
RERANKER.mkdir(parents=True, exist_ok=True)
base_scores = expanded_eval / 'style_embedding_scores.npz'
assert base_scores.exists(), 'Run Part 7 first'

pretrained_dir = RERANKER / 'eval_challenger_pretrained'
run([sys.executable, 'scripts/style_embedding_recall.py', '--input', str(HELDOUT), '--out-dir', str(pretrained_dir), '--model-name', 'Blablablab/multilingual-style-representation', '--model-revision', 'b0147bbf450424fe72c8525fcc02e2e39e3a4024', '--batch-size', '128', '--train-cap', '300', '--eval-splits', 'dev,test', '--device', 'cuda', '--skip-existing'])
pretrained_scores = pretrained_dir / 'style_embedding_scores.npz'

classical_dir = RERANKER / 'eval_classical'
run([sys.executable, 'scripts/style_robust_baseline.py', '--input', str(HELDOUT), '--out-dir', str(classical_dir), '--max-features', '120000', '--skip-existing'])
classical_scores = classical_dir / 'style_robust_scores.npz'

score_specs = [
    f'deployed={base_scores}:single_centroid_scores',
    f'pretrained={pretrained_scores}:single_centroid_scores',
    f'classical={classical_scores}:style_only_fusion',
]
elastic_dir = RERANKER / 'elastic_net_control'
elastic_cmd = [sys.executable, 'scripts/evaluate_multiview_fusion.py', '--input', str(HELDOUT), '--output-dir', str(elastic_dir), '--bootstrap-runs', '5000', '--seed', '20260723']
for spec in score_specs:
    elastic_cmd.extend(['--scores', spec])
run(elastic_cmd)

robust_cmd = [
    sys.executable, 'scripts/evaluate_robust_rank_fusion.py', '--input', str(HELDOUT),
    '--base', 'deployed',
    '--output-dir', str(RERANKER),
    '--weight-step', '0.1',
    '--minimum-base-weight', '0.5',
    '--minimum-group-sources', '8',
    '--group-tolerance', '0.02',
    '--selection-quantile', '0.10',
    '--bootstrap-runs', '5000',
]
for spec in score_specs:
    robust_cmd.extend(['--scores', spec])
run(robust_cmd)

reranker_report_path = RERANKER / 'robust_reranker_metrics.json'
reranker_report = json.loads(reranker_report_path.read_text())
elastic_report_path = elastic_dir / 'multiview_fusion_metrics.json'
elastic_report = json.loads(elastic_report_path.read_text())
display(pd.DataFrame(reranker_report['test_metrics']).T)
print('elastic-net control:', elastic_report['test_metrics'])
print('selected dev weights:', reranker_report['robust_selection']['weights'])
print('paired profile bootstrap:', reranker_report['paired_profile_bootstrap'])
print('decision:', reranker_report['decision'])

summary_path = EXP / 'expansion_summary.json'
summary = json.loads(summary_path.read_text())
summary['reranker_upgrade'] = {
    'metrics': str(reranker_report_path),
    'elastic_net_control': str(elastic_report_path),
    'selected_weights': reranker_report['robust_selection']['weights'],
    'decision': reranker_report['decision'],
    'fusion_adopted': reranker_report['fusion_adopted'],
}
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False))
print('RETURN RERANKER FILES:')
for path in [reranker_report_path, elastic_report_path, RERANKER / 'robust_reranker_candidates.csv', RERANKER / 'robust_reranker_source_scores.npz']:
    print(path)